Helpdesk Process Predictor - Complete version

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Concatenate, Bidirectional
import pickle
import os
import datetime
import matplotlib.pyplot as plt
from tqdm import tqdm  

class HelpdeskPredictor:
    def __init__(self, max_seq_length=10, embedding_dim=50):
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        self.model = None
        self.activity_encoder = LabelEncoder()
        self.feature_encoders = {}
        
    def prepare_data(self, file_path):
        """Load and prepare the data"""
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")
            
        df = pd.read_csv(file_path)
        df['Complete Timestamp'] = pd.to_datetime(df['Complete Timestamp'])
        df = df.sort_values(['Case ID', 'Complete Timestamp'])
        
        ######## LAST STATE ENCODING STRATEGY ########
        # Transform categorical values into numerical ones
        # This encoding captures the current state of each feature

        # Activity encoding
        df['Activity_encoded'] = self.activity_encoder.fit_transform(df['Activity']).astype(np.int32)
        
        # Feature encoding
        categorical_features = ['seriousness', 'service_level', 'service_type', 'workgroup']
        for feature in categorical_features:
            encoder = LabelEncoder()
            df[f'{feature}_encoded'] = encoder.fit_transform(df[feature]).astype(np.int32)
            self.feature_encoders[feature] = encoder

        ######## TIME FEATURE PREPARATION ########
        # Calculate time-related features
        df['time_from_start'] = df.groupby('Case ID')['Complete Timestamp'].transform(
            lambda x: (x - x.iloc[0]).dt.total_seconds() / 3600  # Convert to hours
        )
        df['time_to_end'] = df.groupby('Case ID')['Complete Timestamp'].transform(
            lambda x: (x.iloc[-1] - x).dt.total_seconds() / 3600  # Convert to hours
        )
        df['next_activity_time'] = df.groupby('Case ID')['Complete Timestamp'].transform(
            lambda x: x.shift(-1) - x
        ).dt.total_seconds() / 3600  # Convert to hours

        print(f"################################")
        print(f"\nHelpDesk Event Log Statistics:")
        print(f"Total events: {len(df)}")
        print(f"Unique cases: {df['Case ID'].nunique()}")
        print(f"Unique activities: {df['Activity'].nunique()}")
        print("\nActivity distribution:")
        print(df['Activity'].value_counts())
        
        #print("\nTime Statistics (in hours):")
        #print("\nTime to end statistics:")
        #print(df['time_to_end'].describe())
        #print("\nTime between activities statistics:")
        #print(df['next_activity_time'].describe())
            
        return df
        
    def create_sequences(self, df):
        """Create sequences for training including time features"""
        sequences = []
        next_activities = []
        features = []
        case_ids = []
        time_to_end = []
        time_to_next = []

        ######## CASE-LEVEL ENCODING STRATEGY ########
        # Define features that characterize each case
        # These features remain constant for each case
        encoded_features = ['seriousness_encoded', 'service_level_encoded', 
                          'service_type_encoded', 'workgroup_encoded']

        ######## PREFIX EXTRACTION ########
        for case_id in df['Case ID'].unique():
            case_df = df[df['Case ID'] == case_id]
            activities = case_df['Activity_encoded'].values.astype(np.int32)
            
            for i in range(1, len(activities)):
                # Activity sequences
                seq = activities[max(0, i-self.max_seq_length):i]
                sequences.append(seq)
                next_activities.append(activities[i])
                
                # Features and case ID
                feat = case_df.iloc[i-1][encoded_features].values.astype(np.float32)
                features.append(feat)
                case_ids.append(case_id)
                
                # Time features
                time_to_end.append(case_df.iloc[i]['time_to_end'])
                if i < len(activities) - 1:
                    time_to_next.append(case_df.iloc[i]['next_activity_time'])
                else:
                    time_to_next.append(0)  # For last activity in case

        ######## INDEX-BASED ENCODING STRATEGY ########
        # Transform sequences into fixed-length arrays using padding
        # This preserves the temporal order of events
        X_seq = pad_sequences(sequences, maxlen=self.max_seq_length, 
                            padding='pre', dtype='int32')
        X_feat = np.array(features, dtype='float32')
        y_act = np.array(next_activities, dtype='int32')
        y_time_end = np.array(time_to_end, dtype='float32')
        y_time_next = np.array(time_to_next, dtype='float32')
        
        return X_seq, X_feat, y_act, y_time_end, y_time_next, case_ids

    def build_model(self, n_activities, n_features):
        seq_input = Input(shape=(self.max_seq_length,), dtype='int32', name='sequence_input')
        x = Embedding(input_dim=n_activities, output_dim=self.embedding_dim, name='embedding_layer')(seq_input)
        x = Bidirectional(LSTM(100, return_sequences=True))(x) # Bidirectional LSTM
        x = LSTM(100, name='lstm_layer')(x) # Seconda LSTM
        feat_input = Input(shape=(n_features,), dtype='float32', name='feature_input')
        shared_features = Concatenate(name='concatenate_layer')([x, feat_input])
        act_dense = Dense(200, activation='relu', name='act_dense_1')(shared_features) # Neuroni aumentati
        act_drop = Dropout(0.3, name='act_dropout_1')(act_dense) # Dropout aumentato
        act_dense2 = Dense(100, activation='relu', name='act_dense_2')(act_drop) # Neuroni aumentati
        act_drop2 = Dropout(0.3, name='act_dropout_2')(act_dense2) # Dropout aumentato
        activity_output = Dense(n_activities, activation='softmax', name='activity_output')(act_drop2)
        time_end_dense = Dense(100, activation='relu', name='time_end_dense_1')(shared_features) # Neuroni aumentati
        time_end_drop = Dropout(0.3, name='time_end_dropout_1')(time_end_dense) # Dropout aumentato
        time_to_end_output = Dense(1, name='time_to_end_output')(time_end_drop)
        time_next_dense = Dense(100, activation='relu', name='time_next_dense_1')(shared_features) # Neuroni aumentati
        time_next_drop = Dropout(0.3, name='time_next_dropout_1')(time_next_dense) # Dropout aumentato
        time_to_next_output = Dense(1, name='time_to_next_output')(time_next_drop)
    
        model = Model(
            inputs=[seq_input, feat_input],
            outputs=[activity_output, time_to_end_output, time_to_next_output],
            name='helpdesk_predictor'
        )
    
        model.compile(
            optimizer='adam',
            loss={
                'activity_output': 'sparse_categorical_crossentropy',
                'time_to_end_output': 'mae',
                'time_to_next_output': 'mae'
            },
            metrics={
                'activity_output': [tf.keras.metrics.SparseCategoricalAccuracy(name='Next_Activity_Accuracy')],
                'time_to_end_output': [tf.keras.metrics.MeanAbsoluteError(name='Remaining_Time_MAE')],
                'time_to_next_output': [tf.keras.metrics.MeanAbsoluteError(name='Next_Activity_Time_MAE')]
            }
        )
    
        self.model = model
        return model

    class CustomProgressCallback(tf.keras.callbacks.Callback):
        def __init__(self, epochs):
            super().__init__()
            self.epochs = epochs
            self.pbar = None

        def on_train_begin(self, logs=None):
            self.pbar = tqdm(total=self.epochs, desc='Training', bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} epochs [Remaining Time: {remaining}] {postfix}')

        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            acc = logs.get('activity_output_Next_Activity_Accuracy', 0.0)
            mae_end = logs.get('time_to_end_output_Remaining_Time_MAE', 0.0)
            mae_next = logs.get('time_to_next_output_Next_Activity_Time_MAE', 0.0)
            self.pbar.set_postfix({
                'Next Activity Accuracy (F1-Score)': f'{acc:.4f}',
                'Remaining Process Time (MAE)': f'{mae_end:.4f}',
                'Time to Next Activity (MAE)': f'{mae_next:.4f}'
            })
            self.pbar.update(1)

        def on_train_end(self, logs=None):
            self.pbar.close()

    def train_and_evaluate(self, X_seq, X_feat, y_act, y_time_end, y_time_next, epochs=50, batch_size=32, validation_split=0.2):
        """Train the model and provide detailed evaluation"""
        # Split the data
        indices = np.arange(len(X_seq))
        train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
        
        X_seq_train, X_seq_test = X_seq[train_idx], X_seq[test_idx]
        X_feat_train, X_feat_test = X_feat[train_idx], X_feat[test_idx]
        y_act_train, y_act_test = y_act[train_idx], y_act[test_idx]
        y_time_end_train, y_time_end_test = y_time_end[train_idx], y_time_end[test_idx]
        y_time_next_train, y_time_next_test = y_time_next[train_idx], y_time_next[test_idx]
        
        if self.model is None:
            n_activities = len(self.activity_encoder.classes_)
            n_features = X_feat.shape[1]
            self.build_model(n_activities, n_features)

        print("\n")
        self.model.summary()

        custom_callback = self.CustomProgressCallback(epochs=epochs)
        
        history = self.model.fit(
            [X_seq_train, X_feat_train],
            {
                'activity_output': y_act_train,
                'time_to_end_output': y_time_end_train,
                'time_to_next_output': y_time_next_train
            },
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            verbose=0,
            callbacks=[custom_callback]  
        )
        
        plt.figure(figsize=(15, 5))
        
        # Plot Next Activity Accuracy
        plt.subplot(1, 3, 1)
        plt.plot(history.history['activity_output_Next_Activity_Accuracy'], label='Training')
        plt.plot(history.history['val_activity_output_Next_Activity_Accuracy'], label='Validation')
        plt.title('Next Activity F1-Score')
        plt.xlabel('Epoch')
        plt.ylabel('F1-Score')
        plt.legend()
        
        # Plot Remaining Time MAE
        plt.subplot(1, 3, 2)
        plt.plot(history.history['time_to_end_output_Remaining_Time_MAE'], label='Training')
        plt.plot(history.history['val_time_to_end_output_Remaining_Time_MAE'], label='Validation')
        plt.title('Remaining Process Time (MAE)')
        plt.xlabel('Epoch')
        plt.ylabel('MAE')
        plt.legend()
        
        # Plot Time to Next Activity MAE
        plt.subplot(1, 3, 3)
        plt.plot(history.history['time_to_next_output_Next_Activity_Time_MAE'], label='Training')
        plt.plot(history.history['val_time_to_next_output_Next_Activity_Time_MAE'], label='Validation')
        plt.title('Time to Next Activity MAE')
        plt.xlabel('Epoch')
        plt.ylabel('MAE')
        plt.legend()
        
        plt.tight_layout()
        plt.show()
        
        return history

    def predict_next(self, sequence, features):
        """Predict next activity with time predictions"""
        if not isinstance(sequence, list):
            sequence = [sequence]

        if isinstance(sequence[0], (list, tuple, np.ndarray)):
            sequence = [item for sublist in sequence for item in sublist]
        
        try:
            # Convert sequence to encoded form
            seq_encoded = self.activity_encoder.transform(sequence)
            X_seq = pad_sequences([seq_encoded], maxlen=self.max_seq_length, 
                                padding='pre', dtype='int32')
            X_feat = np.array([features], dtype='float32')
            
            # Get predictions
            activity_pred, time_end_pred, time_next_pred = self.model.predict([X_seq, X_feat])
            
            # Process activity predictions
            pred_probs = activity_pred[0]
            top_indices = pred_probs.argsort()[-3:][::-1]
            predictions = []
            
            for idx in top_indices:
                activity = self.activity_encoder.inverse_transform([idx])[0]
                probability = pred_probs[idx]
                time_to_end = float(time_end_pred[0][0])
                time_to_next = float(time_next_pred[0][0])
                predictions.append({
                    'activity': activity,
                    'probability': probability,
                    'time_to_end': time_to_end,
                    'time_to_next': time_to_next
                })
            
            return predictions
            
        except Exception as e:
            print(f"Error in predict_next: {str(e)}")
            print(f"Sequence shape/content: {np.array(sequence).shape} / {sequence}")
            print(f"Features shape/content: {np.array(features).shape} / {features}")
            raise

    def save_model(self, directory):
        """Save model and parameters"""
        if not os.path.exists(directory):
            os.makedirs(directory)
            
        # Save model in new format
        model_path = os.path.join(directory, 'helpdesk_model.keras')
        self.model.save(model_path)
        
        # Save state
        state = {
            'activity_encoder': self.activity_encoder,
            'feature_encoders': self.feature_encoders,
            'max_seq_length': self.max_seq_length,
            'embedding_dim': self.embedding_dim
        }
        state_path = os.path.join(directory, 'model_state.pkl')
        with open(state_path, 'wb') as f:
            pickle.dump(state, f)

def main():
    try:
        current_dir = os.getcwd()
        data_path = os.path.join(current_dir, 'dataset', 'HelpDesk', 'finale.csv')
        model_save_dir = os.path.join(current_dir, 'models')
        predictor = HelpdeskPredictor()
        df = predictor.prepare_data(data_path)
        X_seq, X_feat, y_act, y_time_end, y_time_next, case_ids = predictor.create_sequences(df)
        history = predictor.train_and_evaluate(X_seq, X_feat, y_act, y_time_end, y_time_next, epochs=50)
        history_df = pd.DataFrame(history.history)
        history_csv_path = os.path.join(model_save_dir, 'training_history.csv')
        history_df.to_csv(history_csv_path, index=False)
        print(f"\nModel Metrics and Weights saved the /models folder")

        if not os.path.exists(model_save_dir):
            os.makedirs(model_save_dir)
        predictor.save_model(model_save_dir)

        np.save(os.path.join(model_save_dir, 'X_seq.npy'), X_seq)
        np.save(os.path.join(model_save_dir, 'X_feat.npy'), X_feat)
        np.save(os.path.join(model_save_dir, 'y_act.npy'), y_act)
        np.save(os.path.join(model_save_dir, 'y_time_end.npy'), y_time_end)
        np.save(os.path.join(model_save_dir, 'y_time_next.npy'), y_time_next)
        df.to_csv(os.path.join(model_save_dir, 'df.csv'), index=False)

    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

################################

HelpDesk Event Log Statistics:
Total events: 21348
Unique cases: 4580
Unique activities: 14

Activity distribution:
Activity
Take in charge ticket    5060
Resolve ticket           4983
Assign seriousness       4938
Closed                   4574
Wait                     1463
Require upgrade           119
Insert ticket             118
Create SW anomaly          67
Resolve SW anomaly         13
Schedule intervention       5
VERIFIED                    3
RESOLVED                    2
INVALID                     2
DUPLICATE                   1
Name: count, dtype: int64




Model: "helpdesk_predictor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_layer     │ (None, 10, 50)    │        700 │ sequence_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 10, 200)   │    120,800 │ embedding_layer[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_layer (LSTM)   │ (None, 100)       │    120,400 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_input       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_layer   │ (None, 104)       │          0 │ lstm_layer[0][0], │
│ (Concatenate)       │                   │            │ feature_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ act_dense_1 (Dense) │ (None, 200)       │     21,000 │ concatenate_laye… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ act_dropout_1       │ (None, 200)       │          0 │ act_dense_1[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ act_dense_2 (Dense) │ (None, 100)       │     20,100 │ act_dropout_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_end_dense_1    │ (None, 100)       │     10,500 │ concatenate_laye… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_next_dense_1   │ (None, 100)       │     10,500 │ concatenate_laye… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ act_dropout_2       │ (None, 100)       │          0 │ act_dense_2[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_end_dropout_1  │ (None, 100)       │          0 │ time_end_dense_1… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_next_dropout_1 │ (None, 100)       │          0 │ time_next_dense_… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activity_output     │ (None, 14)        │      1,414 │ act_dropout_2[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_to_end_output  │ (None, 1)         │        101 │ time_end_dropout… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_to_next_output │ (None, 1)         │        101 │ time_next_dropou… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 305,616 (1.17 MB)

 Trainable params: 305,616 (1.17 MB)

 Non-trainable params: 0 (0.00 B)

Training:  32%|███▏      | 16/50 epochs [Remaining Time: 02:17] , Next Activity Accuracy (F1-Score)=0.7880, Remaining Process Time (MAE)=179.4166, Time to Next Activity (MAE)=164.3307

In the following cell, there are some testing examples:
- Top 3 Predicted Activities vs Actual Activity
- Predicted Remaining Process Time vs Actual Remaining Process Time
- Predicted Time Between 2 Activities vs Actual Time Between 2 Activities

In [12]:
import os
import pickle
import numpy as np
import tensorflow as tf
import pandas as pd

def load_model_and_state(directory):
    model_path = os.path.join(directory, 'helpdesk_model.keras')
    state_path = os.path.join(directory, 'model_state.pkl')
    model = tf.keras.models.load_model(model_path)
    with open(state_path, 'rb') as f:
        state = pickle.load(f)
    return model, state

model_save_dir = os.path.join(os.getcwd(), 'models')
model, state = load_model_and_state(model_save_dir)

class HelpdeskPredictor:
    def __init__(self, max_seq_length=10, embedding_dim=50):
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        self.model = None
        self.activity_encoder = None
        self.feature_encoders = None

    def predict_next(self, sequence, features):
        if not isinstance(sequence, list):
            sequence = [sequence]
        if isinstance(sequence[0], (list, tuple, np.ndarray)):
            sequence = [item for sublist in sequence for item in sublist]
        try:
            seq_encoded = self.activity_encoder.transform(sequence)
            X_seq = tf.keras.preprocessing.sequence.pad_sequences([seq_encoded], maxlen=self.max_seq_length,
                                                                   padding='pre', dtype='int32')
            X_feat = np.array([features], dtype='float32')
            activity_pred, time_end_pred, time_next_pred = self.model.predict([X_seq, X_feat])
            pred_probs = activity_pred[0]
            top_indices = pred_probs.argsort()[-3:][::-1]
            predictions = []
            for idx in top_indices:
                activity = self.activity_encoder.inverse_transform([idx])[0]
                probability = pred_probs[idx]
                time_to_end = float(time_end_pred[0][0])
                time_to_next = float(time_next_pred[0][0])
                predictions.append({
                    'activity': activity,
                    'probability': probability,
                    'time_to_end': time_to_end,
                    'time_to_next': time_to_next
                })
            return predictions
        except Exception as e:
            print(f"Error in predict_next: {str(e)}")
            print(f"Sequence shape/content: {np.array(sequence).shape} / {sequence}")
            print(f"Features shape/content: {np.array(features).shape} / {features}")
            raise

predictor = HelpdeskPredictor(state['max_seq_length'], state['embedding_dim'])
predictor.model = model
predictor.activity_encoder = state['activity_encoder']
predictor.feature_encoders = state['feature_encoders']

X_seq = np.load(os.path.join(model_save_dir, 'X_seq.npy'))
X_feat = np.load(os.path.join(model_save_dir, 'X_feat.npy'))
y_act = np.load(os.path.join(model_save_dir, 'y_act.npy'))
y_time_end = np.load(os.path.join(model_save_dir, 'y_time_end.npy'))
y_time_next = np.load(os.path.join(model_save_dir, 'y_time_next.npy'))
df = pd.read_csv(os.path.join(model_save_dir, 'df.csv'))
df['Complete Timestamp'] = pd.to_datetime(df['Complete Timestamp'])

def color_text(text, color_code):
    return f"\033[{color_code}m{text}\033[0m"

def analyze_real_examples(predictor, df, num_cases=1):
    random_cases = np.random.choice(df['Case ID'].unique(), num_cases, replace=False)
    for case_id in random_cases:
        case_df = df[df['Case ID'] == case_id].copy()
        case_df = case_df.sort_values('Complete Timestamp')
        activities = case_df['Activity'].tolist()
        print(f"\n###### Testing a Random Case ID ######")
        print(f"CASE ID: {case_id}")
        durations = [(activities[i], (case_df['Complete Timestamp'].iloc[i+1] - case_df['Complete Timestamp'].iloc[i]).total_seconds() / 3600)
                     for i in range(len(activities) - 1)]
        total_time = (case_df['Complete Timestamp'].iloc[-1] - case_df['Complete Timestamp'].iloc[0]).total_seconds() / 3600
        print(f"List of activities order: {', '.join(activities)}")
        print(f"List of activities duration: {', '.join([f'{act} ({dur:.2f} hours)' for act, dur in durations])}")
        print(f"Process total time: {total_time:.2f} hours")
        print(f"#############################################")
        for i in range(1, len(activities)):
            current_sequence = activities[:i]
            actual_next = activities[i]
            feature_values = case_df.iloc[i - 1][[f'{feat}_encoded' for feat in
                                                     ['seriousness', 'service_level', 'service_type', 'workgroup']]].values
            real_time_to_end = case_df.iloc[i]['time_to_end'] if i < len(activities) else 0
            real_time_to_next = case_df.iloc[i]['next_activity_time'] if i < len(activities) - 1 else 0

            print(f"\n1. Prediction of next activity ###############")
            print(f"Current sequence: {' -> '.join(current_sequence)}")
            print(f"Actual next activity: {actual_next}")
            try:
                predictions = predictor.predict_next(current_sequence, feature_values)
                print("Top 3 predictions (Activity - %):")
                for pred in predictions:
                    correct_mark = "✓" if pred['activity'] == actual_next else " "
                    color_code = "32" if pred['activity'] == actual_next else "31"
                    colored_activity = color_text(f"{pred['activity']}: {pred['probability']:.2f}", color_code)
                    print(f"    {correct_mark} {colored_activity}")

                print(f"\n2. Predicted time to end ###############")
                time_end_color = "32" if abs(predictions[0]['time_to_end'] - real_time_to_end) < 1 else "31"
                colored_time_end_actual = color_text(f"{real_time_to_end:.2f} hours (Actual value)", time_end_color)
                colored_time_end_pred = color_text(f"{predictions[0]['time_to_end']:.2f} hours (Predicted value)", time_end_color)
                print(f"from {activities[i-1]} to end: {colored_time_end_actual}")
                print(f"from {activities[i-1]} to end: {colored_time_end_pred}")

                print(f"\n3. Predicted time to next ###############")
                if i < len(activities) - 1 and pd.notna(real_time_to_next):
                    real_time_next_act = case_df.iloc[i]['next_activity_time'] if i < len(activities)-1 else 0
                    time_next_color = "32" if abs(predictions[0]['time_to_next'] - real_time_next_act) < 1 else "31"
                    colored_time_next_actual = color_text(f"{real_time_next_act:.2f} hours (Actual value)", time_next_color)
                    colored_time_next_pred = color_text(f"{predictions[0]['time_to_next']:.2f} hours (Predicted value)", time_next_color)
                    print(f"from {activities[i-1]} to {activities[i]}: {colored_time_next_actual}")
                    print(f"from {activities[i-1]} to {activities[i]}: {colored_time_next_pred}")
                else:
                    print("Time to next is the same as time to end for the last activity.")
                print("-" * 50)
            except Exception as e:
                print(f"Error during prediction: {str(e)}")
                continue

try:
    analyze_real_examples(predictor, df)
except Exception as e:
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()


###### Testing a Random Case ID ######
CASE ID: Case 1209
List of activities order: Assign seriousness, Take in charge ticket, Resolve ticket, Closed
List of activities duration: Assign seriousness (21.84 hours), Take in charge ticket (2.15 hours), Resolve ticket (1248.04 hours)
Process total time: 1272.03 hours
#############################################

1. Prediction of next activity ###############
Current sequence: Assign seriousness
Actual next activity: Take in charge ticket
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
Top 3 predictions (Activity - %):
    ✓ Take in charge ticket: 0.78
      Assign seriousness: 0.12
      Resolve ticket: 0.07

2. Predicted time to end ###############
from Assign seriousness to end: 1250.19 hours (Actual value)
from Assign seriousness to end: 884.07 hours (Predicted value)

3. Predicted time to next ###############
from Assign seriousness to Take in charge ticket: 2.15 hours (Actual value)
from Assign seriousness to Take in charge ticket: 39.37 hour